In [1]:
import warnings

warnings.filterwarnings("ignore")
from torch import multiprocessing

from collections import defaultdict

import matplotlib.pyplot as plt
import torch
from tensordict.nn import TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor
from torch import nn
from torchrl.collectors import Collector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (
    Compose,
    DoubleToFloat,
    ObservationNorm,
    StepCounter,
    TransformedEnv,
)
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator, MaskedCategorical
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm

from env import ZappyEnv

import csv

In [2]:
is_fork = multiprocessing.get_start_method() == "fork"
device = (
    torch.device(0)
    if torch.cuda.is_available() and not is_fork
    else torch.device("cpu")
)

In [3]:
## Hyperparameters

num_cells = 256  # number of cells in each layer i.e. output dim.
lr = 3e-4
max_grad_norm = 1.0

## Environment parameters

frames_per_batch = 200  # number of frames to collect for each PPO update
# For a complete training, bring the number of frames up to 1M
total_frames = 100_000  # number of frames to train on before stopping

## PPO parameters

sub_batch_size = 64  # cardinality of the sub-samples gathered from the current data in the inner loop
num_epochs = 10  # optimization steps per batch of data collected
clip_epsilon = 0.2  # clip value for PPO loss
gamma = 0.99
lmbda = 0.95  # GAE lambda parameter
entropy_eps = 0.01  # coefficient for the entropy bonus, see the PPO loss equation in the intro for more context.

In [4]:
base_env = ZappyEnv()

In [5]:
env = TransformedEnv(
    base_env,
    Compose(
        # normalize observations
        ObservationNorm(in_keys=["observation"]),
        DoubleToFloat(),
        StepCounter(),
    ),
)

In [6]:
try:
    base_env.reset()
    env.transform[0].init_stats(num_iter=10, reduce_dim=0, cat_dim=0)
except (TimeoutError, ConnectionError) as exc:
    raise RuntimeError(
        "init_stats failed: make sure zappy_server is running and responsive before collecting normalization stats."
    ) from exc

In [7]:
print("normalization constant shape:", env.transform[0].loc.shape)

normalization constant shape: torch.Size([64])


In [8]:
print("observation_spec:", env.observation_spec)
print("reward_spec:", env.reward_spec)
print("input_spec:", env.input_spec)
print("action_spec (as defined by input_spec):", env.action_spec)

observation_spec: Composite(
    observation: UnboundedContinuous(
        shape=torch.Size([64]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([64]), device=cpu, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([64]), device=cpu, dtype=torch.float32, contiguous=True)),
        device=cpu,
        dtype=torch.float32,
        domain=continuous),
    step_count: BoundedDiscrete(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, contiguous=True),
            high=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, contiguous=True)),
        device=cpu,
        dtype=torch.int64,
        domain=discrete),
    device=None,
    shape=torch.Size([]),
    data_cls=None)
reward_spec: UnboundedContinuous(
    shape=torch.Size([1]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, contiguous=True)

In [9]:
print("position:", env.position)

position: (10, 10)


In [10]:
print("position:", getattr(env, "position", (0, 0)))

position: (10, 10)


In [11]:
rollout = env.rollout(3)
print("rollout of three steps:", rollout)
print("Shape of the rollout TensorDict:", rollout.batch_size)

rollout of three steps: TensorDict(
    fields={
        action: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.int64, is_shared=False),
        action_mask: Tensor(shape=torch.Size([3, 23]), device=cpu, dtype=torch.bool, is_shared=False),
        done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                action_mask: Tensor(shape=torch.Size([3, 23]), device=cpu, dtype=torch.bool, is_shared=False),
                done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([3, 64]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                step_count: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.int64, is_shared=False),
                terminated: Tensor(shape=torch.Size([3, 1]), d

In [12]:
actor_net = nn.Sequential(
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(23, device=device),
)

In [13]:
policy_module = TensorDictModule(
    actor_net, in_keys=["observation"], out_keys=["logits"]
)

In [14]:
policy_module = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec,
    in_keys={"logits": "logits", "mask": "action_mask"},
    distribution_class=MaskedCategorical,
    return_log_prob=True,
)

In [15]:
value_net = nn.Sequential(
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(1, device=device),
)

value_module = ValueOperator(
    module=value_net,
    in_keys=["observation"],
)

In [16]:
print("Running policy:", policy_module(env.reset()))
print("Running value:", value_module(env.reset()))

Running policy: TensorDict(
    fields={
        action: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.int64, is_shared=False),
        action_log_prob: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.float32, is_shared=False),
        action_mask: Tensor(shape=torch.Size([23]), device=cpu, dtype=torch.bool, is_shared=False),
        done: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        logits: Tensor(shape=torch.Size([23]), device=cpu, dtype=torch.float32, is_shared=False),
        observation: Tensor(shape=torch.Size([64]), device=cpu, dtype=torch.float32, is_shared=False),
        step_count: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, is_shared=False),
        terminated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)
Running value: TensorDict(
    fields={
        action_mask: Tensor(shape=torch.Size([23]), device=c

In [17]:
collector = Collector(
    env,
    policy_module,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
    split_trajs=False,
    device=device,
)

In [18]:
replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=frames_per_batch),
    sampler=SamplerWithoutReplacement(),
)

In [19]:
advantage_module = GAE(
    gamma=gamma,
    lmbda=lmbda,
    value_network=value_module,
    average_gae=True,
    device=device,
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=clip_epsilon,
    entropy_bonus=bool(entropy_eps),
    entropy_coeff=entropy_eps,
    # these keys match by default but we set this for completeness
    critic_coeff=1.0,
    loss_critic_type="smooth_l1",
)

optim = torch.optim.Adam(loss_module.parameters(), lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim, total_frames // frames_per_batch, 0.0
)

In [20]:
import time
t0 = time.time()
td = env.reset()
print(f"reset en {time.time()-t0:.2f}s")

reset en 0.09s


In [21]:
def format_command_trace(commands, limit=12):
    if not commands:
        return "[]"
    if len(commands) <= limit:
        return "[" + " -> ".join(commands) + "]"
    head = " -> ".join(commands[: limit // 2])
    tail = " -> ".join(commands[-limit // 2 :])
    return f"[{head} -> ... -> {tail}]"

In [22]:
logs = defaultdict(list)
pbar = tqdm(total=total_frames)
eval_str = ""
csv_path = "logs.csv"
last_eval_reward_mean = ""
last_eval_reward_sum = ""
last_eval_step_count = ""
last_eval_command_trace = ""

# We iterate over the collector until it reaches the total number of frames it was
# designed to collect:
with open(csv_path, "w", newline="") as csvfile:
    fieldnames = [
        "batch",
        "reward",
        "eval reward",
        "eval reward (sum)",
        "step_count",
        "lr",
        "commands",
        "eval commands",
    ]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for i, tensordict_data in enumerate(collector):
        batch_idx = i + 1
        batch_commands = [
            base_env._action_to_cmd(int(action))
            for action in tensordict_data["action"].reshape(-1).tolist()
        ]
        command_trace = format_command_trace(batch_commands)

        # we now have a batch of data to work with. Let's learn something from it.
        for _ in range(num_epochs):
            # We'll need an "advantage" signal to make PPO work.
            # We re-compute it at each epoch as its value depends on the value
            # network which is updated in the inner loop.
            advantage_module(tensordict_data)
            data_view = tensordict_data.reshape(-1)
            replay_buffer.extend(data_view.cpu())
            for _ in range(frames_per_batch // sub_batch_size):
                subdata = replay_buffer.sample(sub_batch_size)
                loss_vals = loss_module(subdata.to(device))
                loss_value = (
                    loss_vals["loss_objective"]
                    + loss_vals["loss_critic"]
                    + loss_vals["loss_entropy"]
                )

                # Optimization: backward, grad clipping and optimization step
                loss_value.backward()
                # this is not strictly mandatory but it's good practice to keep
                # your gradient norm bounded
                torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_grad_norm)
                optim.step()
                optim.zero_grad()

        current_reward = tensordict_data["next", "reward"].mean().item()
        current_step_count = tensordict_data["step_count"].max().item()
        current_lr = optim.param_groups[0]["lr"]

        logs["reward"].append(current_reward)
        logs["step_count"].append(current_step_count)
        logs["lr"].append(current_lr)
        pbar.update(tensordict_data.numel())

        cum_reward_str = (
            f"average reward={current_reward: 4.4f} (init={logs['reward'][0]: 4.4f})"
        )
        stepcount_str = f"step count (max): {current_step_count}"
        lr_str = f"lr policy: {current_lr: 4.4f}"

        if i % 10 == 0:
            # We evaluate the policy once every 10 batches of data.
            # Evaluation is rather simple: execute the policy without exploration
            # (take the expected value of the action distribution) for a given
            # number of steps (1000, which is our ``env`` horizon).
            # The ``rollout`` method of the ``env`` can take a policy as argument:
            # it will then execute this policy at each step.
            with set_exploration_type(ExplorationType.DETERMINISTIC), torch.no_grad():
                # execute a rollout with the trained policy
                eval_rollout = env.rollout(1000, policy_module)
                current_eval_reward = eval_rollout["next", "reward"].mean().item()
                current_eval_reward_sum = eval_rollout["next", "reward"].sum().item()
                current_eval_step_count = eval_rollout["step_count"].max().item()
                eval_commands = [
                    base_env._action_to_cmd(int(action))
                    for action in eval_rollout["action"].reshape(-1).tolist()
                ]
                eval_command_trace = format_command_trace(eval_commands)
                logs["eval reward"].append(current_eval_reward)
                logs["eval reward (sum)"].append(current_eval_reward_sum)
                logs["eval step_count"].append(current_eval_step_count)
                eval_str = (
                    f"eval reward={current_eval_reward: 4.4f}, "
                    f"eval cumulative reward={current_eval_reward_sum: 4.4f}, "
                    f"eval step-count={current_eval_step_count}, "
                    f"commands={eval_command_trace}"
                )
                last_eval_reward_mean = current_eval_reward
                last_eval_reward_sum = current_eval_reward_sum
                last_eval_step_count = current_eval_step_count
                last_eval_command_trace = eval_command_trace
                del eval_rollout
            print(
                f"[eval] batch {batch_idx:04d} | "
                f"reward={current_reward: 4.4f} | "
                f"eval_reward={current_eval_reward: 4.4f} | "
                f"eval_sum={current_eval_reward_sum: 4.4f} | "
                f"eval_step_count={current_eval_step_count} | "
                f"commands={command_trace} | "
                f"eval_commands={eval_command_trace}",
                flush=True,
            )
        else:
            print(
                f"[train] batch {batch_idx:04d} | "
                f"reward={current_reward: 4.4f} | "
                f"step_count={current_step_count} | "
                f"lr={current_lr: 4.4f} | "
                f"commands={command_trace}",
                flush=True,
            )

        writer.writerow(
            {
                "batch": batch_idx,
                "reward": current_reward,
                "eval reward": last_eval_reward_mean,
                "eval reward (sum)": last_eval_reward_sum,
                "step_count": current_step_count,
                "lr": current_lr,
                "commands": command_trace,
                "eval commands": last_eval_command_trace,
            }
        )
        csvfile.flush()

        pbar.set_description(
            ", ".join([eval_str, cum_reward_str, stepcount_str, lr_str])
        )

        # We're also using a learning rate scheduler. Like the gradient clipping,
        # this is a nice-to-have but nothing necessary for PPO to work.
        scheduler.step()

  0%|          | 0/100000 [00:00<?, ?it/s]

2026-06-04 15:34:03,346 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([200]) shape [END]


  0%|          | 200/100000 [02:37<21:48:44,  1.27it/s]

[eval] batch 0001 | reward=-10.7282 | eval_reward=-9.9338 | eval_sum=-764.8999 | eval_step_count=76 | commands=[Right -> Right -> Inventory -> Left -> Left -> Fork -> ... -> Eject -> Forward -> Take phiras -> Set food -> Set food -> Take deraumere] | eval_commands=[Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> ... -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere]


eval reward=-9.9338, eval cumulative reward=-764.8999, eval step-count=76, commands=[Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> ... -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere], average reward=-10.7282 (init=-10.7282), step count (max): 61, lr policy:  0.0003:   0%|          | 400/100000 [05:31<23:05:53,  1.20it/s]

...

[train] batch 0005 | reward=-10.7615 | step_count=74 | lr= 0.0003 | commands=[Forward -> Fork -> Look -> Take deraumere -> Inventory -> Left -> ... -> Right -> Set food -> Look -> Fork -> Right -> Eject]


eval reward=-9.9338, eval cumulative reward=-764.8999, eval step-count=76, commands=[Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> ... -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere], average reward=-10.7615 (init=-10.7282), step count (max): 74, lr policy:  0.0003:   1%|          | 1200/100000 [16:05<21:27:47,  1.28it/s]

[train] batch 0006 | reward=-8.3062 | step_count=66 | lr= 0.0003 | commands=[Right -> Forward -> Broadcast rally -> Forward -> Set food -> Right -> ... -> Eject -> Set food -> Take food -> Right -> Eject -> Look]


eval reward=-9.9338, eval cumulative reward=-764.8999, eval step-count=76, commands=[Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> ... -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere -> Take deraumere], average reward=-8.3062 (init=-10.7282), step count (max): 66, lr policy:  0.0003:   1%|          | 1200/100000 [16:05<21:27:47,  1.28it/s] 

ConnectionRefusedError: [Errno 61] Connection refused

In [ ]:
plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.plot(logs["reward"])
plt.title("training rewards (average)")
plt.subplot(2, 2, 2)
plt.plot(logs["step_count"])
plt.title("Max step count (training)")
plt.subplot(2, 2, 3)
plt.plot(logs["eval reward (sum)"])
plt.title("Return (test)")
plt.subplot(2, 2, 4)
plt.plot(logs["eval step_count"])
plt.title("Max step count (test)")
plt.show()

In [ ]:
# save agent
torch.save(policy_module.state_dict(), "ppo_policy.pth")
torch.save(value_module.state_dict(), "ppo_value.pth")